In [ ]:
%%capture
if 'google.colab' in str(get_ipython()):
    !pip install --upgrade xee

In [ ]:
import geopandas as gpd
import rasterio
import glob
import rasterio
import numpy as np
import os
import re
import glob
import pandas as pd
import numpy as np
from rasterio.plot import show
from rasterio.mask import mask
from shapely.geometry import Point
from statsmodels.stats.outliers_influence import variance_inflation_factor
from tqdm import tqdm  # For progress tracking

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import rasterio
import numpy as np
import os
import re
import glob

# =========================
# Paths
# =========================
base_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs"
dbh_path = os.path.join(base_path, "DBH_cm")
agb_out = os.path.join(base_path, "AGB_Chave_kg")
os.makedirs(agb_out, exist_ok=True)

# =========================
# Get input files
# =========================
dbh_files = glob.glob(os.path.join(dbh_path, "DBH_Predicted_*_cm.tif"))
h_files = glob.glob(os.path.join(base_path, "H_Predicted_*.tif"))

# =========================
# Parse H files into year ranges
# =========================
h_ranges = []

for f in h_files:
    match = re.search(r"(\d{4})_(\d{4})", os.path.basename(f))
    if match:
        start, end = int(match.group(1)), int(match.group(2))
        h_ranges.append((start, end, f))
    else:
        print(f" Skipped H file (no year range): {os.path.basename(f)}")

print("Loaded H year ranges:")
for r in h_ranges:
    print(f"{r[0]}–{r[1]} → {os.path.basename(r[2])}")

# =========================
# AGB calculation loop
# =========================
for dbh_file in sorted(dbh_files):

    match = re.search(r"DBH_Predicted_(\d{4})_cm", os.path.basename(dbh_file))
    if not match:
        print(f" Skipped DBH file (no year): {os.path.basename(dbh_file)}")
        continue

    dbh_year = int(match.group(1))

    # Find matching H file
    h_file = None
    for start, end, hf in h_ranges:
        if start <= dbh_year <= end:
            h_file = hf
            break

    if h_file is None:
        print(f"No matching H file for DBH year {dbh_year}")
        continue

    with rasterio.open(dbh_file) as dsrc, rasterio.open(h_file) as hsrc:

        D = dsrc.read(1).astype("float32")   # DBH in cm
        H = hsrc.read(1).astype("float32")   # Height in m

        meta = dsrc.meta.copy()
        meta.update(dtype="float32", nodata=0)

        mask = (D > 0) & (H > 0)

        AGB = np.zeros_like(D, dtype="float32")
        AGB[mask] = 0.0673 * (0.598 * (D[mask] ** 2) * H[mask]) ** 0.976

        out_file = os.path.join(agb_out, f"AGB_Chave_kg_{dbh_year}.tif")

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(AGB, 1)

    print(f" AGB calculated for DBH year {dbh_year}")

print("AGB calculation completed successfully (kg per pixel).")


 Skipped H file (no year range): H_Predicted_Masked_2010.tif
 Skipped H file (no year range): H_Predicted_Masked_2005.tif
 Skipped H file (no year range): H_Predicted_Masked_2000.tif
 Skipped H file (no year range): H_Predicted_Masked_1995.tif
 Skipped H file (no year range): H_Predicted_Masked_1990.tif
 Skipped H file (no year range): H_Predicted_2020_Masked.tif
 Skipped H file (no year range): H_Predicted_2015_Masked.tif
 Skipped H file (no year range): H_Predicted_2010_Masked.tif
 Skipped H file (no year range): H_Predicted_2000_Masked.tif
 Skipped H file (no year range): H_Predicted_1995_Masked.tif
 Skipped H file (no year range): H_Predicted_1990_Masked.tif
 Skipped H file (no year range): H_Predicted_2005_Masked.tif
 Skipped H file (no year range): H_Predicted_2024_Masked.tif
Loaded H year ranges:
2019–2020 → H_Predicted_2019_2020.tif
2014–2015 → H_Predicted_2014_2015.tif
2024–2025 → H_Predicted_2024_2025.tif
1989–1990 → H_Predicted_1989_1990.tif
1994–1995 → H_Predicted_1994_1995

In [ ]:
import rasterio
import numpy as np
import os
import glob

# Paths
agb_kg_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg"
agb_tha_out = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_t_ha"
os.makedirs(agb_tha_out, exist_ok=True)

# Conversion factor: kg per pixel → t/ha
# Pixel area = 25 m² → 400 pixels per hectare → multiply by 0.4
conversion_factor = 0.4

# Get all AGB kg files
agb_files = glob.glob(os.path.join(agb_kg_path, "AGB_Chave_kg_*.tif"))

for f in sorted(agb_files):
    with rasterio.open(f) as src:
        agb_kg = src.read(1).astype("float32")
        meta = src.meta.copy()
        meta.update(dtype="float32", nodata=0)

        # Convert to t/ha
        agb_t_ha = agb_kg * conversion_factor

        out_file = os.path.join(agb_tha_out, os.path.basename(f).replace("kg", "t_ha"))

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(agb_t_ha, 1)

    print(f"Converted {os.path.basename(f)} to t/ha")

print(" All AGB rasters converted to t/ha successfully.")


Converted AGB_Chave_kg_1990.tif to t/ha
Converted AGB_Chave_kg_1995.tif to t/ha
Converted AGB_Chave_kg_2000.tif to t/ha
Converted AGB_Chave_kg_2005.tif to t/ha
Converted AGB_Chave_kg_2010.tif to t/ha
Converted AGB_Chave_kg_2015.tif to t/ha
Converted AGB_Chave_kg_2020.tif to t/ha
Converted AGB_Chave_kg_2024.tif to t/ha
 All AGB rasters converted to t/ha successfully.


In [ ]:
# Paths
agb_tha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_t_ha"

# Get all AGB t/ha files
agb_files = sorted(glob.glob(os.path.join(agb_tha_path, "AGB_Chave_t_ha_*.tif")))

# Prepare summary list
summary_list = []

for f in agb_files:
    year = os.path.basename(f).split("_")[-1].replace(".tif","")

    with rasterio.open(f) as src:
        data = src.read(1)
        data = data[data > 0]  # Mask out zeros/nodata

        summary_list.append({
            "Year": year,
            "Mean_t_ha": np.mean(data),
            "Median_t_ha": np.median(data),
            "Min_t_ha": np.min(data),
            "Max_t_ha": np.max(data),
            "Std_t_ha": np.std(data)
        })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_list)
summary_df = summary_df.sort_values("Year").reset_index(drop=True)

# Display
print(summary_df)

   Year  Mean_t_ha  Median_t_ha  Min_t_ha  Max_t_ha  Std_t_ha
0  1990     651.76       651.50    175.05  1,494.91    179.53
1  1995     650.24       646.53    170.49  1,343.69    168.17
2  2000     667.70       650.84    168.56  1,735.70    188.69
3  2005     400.21       171.31     92.89  1,311.66    267.12
4  2010     658.74       653.04    159.45  1,495.26    171.42
5  2015     534.04       418.72    257.40  2,016.22    174.93
6  2020     552.49       418.72    256.35  2,016.22    207.29
7  2024     545.62       418.72    256.64  2,016.22    195.01


Recalculate AGB in MG/H

In [ ]:
import rasterio
import numpy as np
import os
import glob

# Paths
agb_kg_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg"
agb_mgha_out = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_Mg_ha"
os.makedirs(agb_mgha_out, exist_ok=True)

# Conversion factor
# Pixel area ~25 m² → 400 pixels per hectare
# 1 Mg = 1000 kg
# kg/pixel → Mg/ha = kg * 0.4 / 1000 = 0.0004? Let's compute carefully:

# Step 1: 1 pixel = 25 m² → 1 ha = 10,000 m² → 400 pixels/ha
# Step 2: kg → Mg = divide by 1000
# So factor = 400 / 1000 = 0.4 ✅

conversion_factor = 0.4  # kg/pixel → Mg/ha

# Get all AGB kg files
agb_files = sorted(glob.glob(os.path.join(agb_kg_path, "AGB_Chave_kg_*.tif")))

for f in agb_files:
    with rasterio.open(f) as src:
        agb_kg = src.read(1).astype("float32")
        meta = src.meta.copy()
        meta.update(dtype="float32", nodata=0)

        # Convert to Mg/ha
        agb_mgha = agb_kg * conversion_factor

        out_file = os.path.join(agb_mgha_out, os.path.basename(f).replace("kg", "Mg_ha"))

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(agb_mgha, 1)

    print(f"✅ Converted {os.path.basename(f)} to Mg/ha")

print("🎉 All AGB rasters converted to Mg/ha successfully.")


✅ Converted AGB_Chave_kg_1990.tif to Mg/ha
✅ Converted AGB_Chave_kg_1995.tif to Mg/ha
✅ Converted AGB_Chave_kg_2000.tif to Mg/ha
✅ Converted AGB_Chave_kg_2005.tif to Mg/ha
✅ Converted AGB_Chave_kg_2010.tif to Mg/ha
✅ Converted AGB_Chave_kg_2015.tif to Mg/ha
✅ Converted AGB_Chave_kg_2020.tif to Mg/ha
✅ Converted AGB_Chave_kg_2024.tif to Mg/ha
🎉 All AGB rasters converted to Mg/ha successfully.


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# Paths
agb_mgha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_Mg_ha"

# Get all AGB Mg/ha files
agb_files = sorted(glob.glob(os.path.join(agb_mgha_path, "AGB_Chave_Mg_ha_*.tif")))

# Prepare summary list
summary_list = []

for f in agb_files:
    year = os.path.basename(f).split("_")[-1].replace(".tif","")

    with rasterio.open(f) as src:
        data = src.read(1)
        data = data[data > 0]  # Mask out zeros/nodata

        summary_list.append({
            "Year": year,
            "Mean_Mg_ha": np.mean(data),
            "Median_Mg_ha": np.median(data),
            "Min_Mg_ha": np.min(data),
            "Max_Mg_ha": np.max(data),
            "Std_Mg_ha": np.std(data)
        })

# Convert to DataFrame
summary_df1 = pd.DataFrame(summary_list)
summary_df1 = summary_df.sort_values("Year").reset_index(drop=True)

# Display
print(summary_df1)


   Year  Mean_Mg_ha  Median_Mg_ha   Min_Mg_ha    Max_Mg_ha   Std_Mg_ha
0  1990  965.019897    964.632935  259.186371  2213.420898  265.816803
1  1995  962.775879    957.279724  252.434921  1989.515869  249.006104
2  2000  988.627441    963.665161  249.580734  2569.949951  279.386353
3  2005  592.563721    253.649933  137.532440  1942.092041  395.506592
4  2010  975.355896    966.913208  236.092850  2213.947266  253.807327
5  2015  790.727173    619.979492  381.124146  2985.297119  259.011841
6  2020  818.043884    619.979492  379.567169  2985.297119  306.927032
7  2024  807.874023    619.979492  379.997375  2985.297119  288.735107


In [ ]:
import glob
import os
import rasterio
import numpy as np

# Path to the directory containing rasters
agb_mgha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_Mg_ha"

# Get all AGB Mg/ha files
agb_files = sorted(glob.glob(os.path.join(agb_mgha_path, "AGB_Chave_Mg_ha_*.tif")))

# Process each raster
for input_path in agb_files:
    with rasterio.open(input_path) as src:
        # Read the data
        data = src.read(1)
        profile = src.profile

        # Multiply by 0.47
        data_multiplied = data * 0.47

        # Prepare output filename
        dir_name, filename = os.path.split(input_path)
        name, ext = os.path.splitext(filename)
        output_filename = f"{name}_multiplied{ext}"
        output_path = os.path.join(dir_name, output_filename)

        # Save the new raster
        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(data_multiplied, 1)

print("Processing completed. All rasters multiplied by 0.47 and saved.")

Processing completed. All rasters multiplied by 0.47 and saved.


In [ ]:
import rasterio
import numpy as np
import os
import re
import glob

# =========================
# Paths
# =========================
base_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs"
dbh_path = os.path.join(base_path, "DBH_cm")
agb_out_mg = os.path.join(base_path, "AGB_Chave_mg")
os.makedirs(agb_out_mg, exist_ok=True)

# =========================
# Get input files
# =========================
dbh_files = glob.glob(os.path.join(dbh_path, "DBH_Predicted_*_cm.tif"))
h_files = glob.glob(os.path.join(base_path, "H_Predicted_*.tif"))

# =========================
# Parse H files into year ranges
# =========================
h_ranges = []

for f in h_files:
    match = re.search(r"(\d{4})_(\d{4})", os.path.basename(f))
    if match:
        start, end = int(match.group(1)), int(match.group(2))
        h_ranges.append((start, end, f))
    else:
        print(f" Skipped H file (no year range): {os.path.basename(f)}")

print("Loaded H year ranges:")
for r in h_ranges:
    print(f"{r[0]}–{r[1]} → {os.path.basename(r[2])}")

# =========================
# AGB calculation loop in mg
# =========================
for dbh_file in sorted(dbh_files):

    match = re.search(r"DBH_Predicted_(\d{4})_cm", os.path.basename(dbh_file))
    if not match:
        print(f" Skipped DBH file (no year): {os.path.basename(dbh_file)}")
        continue

    dbh_year = int(match.group(1))

    # Find matching H file
    h_file = None
    for start, end, hf in h_ranges:
        if start <= dbh_year <= end:
            h_file = hf
            break

    if h_file is None:
        print(f"No matching H file for DBH year {dbh_year}")
        continue

    with rasterio.open(dbh_file) as dsrc, rasterio.open(h_file) as hsrc:

        D = dsrc.read(1).astype("float32")   # DBH in cm
        H = hsrc.read(1).astype("float32")   # Height in m

        meta = dsrc.meta.copy()
        meta.update(dtype="float32", nodata=0)

        mask = (D > 0) & (H > 0)

        AGB_kg = np.zeros_like(D, dtype="float32")
        AGB_kg[mask] = 0.0673 * (0.598 * (D[mask] ** 2) * H[mask]) ** 0.976

        # Convert kg → mg
        AGB_mg = AGB_kg * 1_000_000

        out_file = os.path.join(agb_out_mg, f"AGB_Chave_mg_{dbh_year}.tif")

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(AGB_mg, 1)

    print(f"✅ AGB calculated for DBH year {dbh_year} in mg/pixel")

print("🎉 All AGB rasters completed successfully (mg per pixel).")


 Skipped H file (no year range): H_Predicted_Masked_2010.tif
 Skipped H file (no year range): H_Predicted_Masked_2005.tif
 Skipped H file (no year range): H_Predicted_Masked_2000.tif
 Skipped H file (no year range): H_Predicted_Masked_1995.tif
 Skipped H file (no year range): H_Predicted_Masked_1990.tif
 Skipped H file (no year range): H_Predicted_2020_Masked.tif
 Skipped H file (no year range): H_Predicted_2015_Masked.tif
 Skipped H file (no year range): H_Predicted_2010_Masked.tif
 Skipped H file (no year range): H_Predicted_2000_Masked.tif
 Skipped H file (no year range): H_Predicted_1995_Masked.tif
 Skipped H file (no year range): H_Predicted_1990_Masked.tif
 Skipped H file (no year range): H_Predicted_2005_Masked.tif
 Skipped H file (no year range): H_Predicted_2024_Masked.tif
Loaded H year ranges:
2019–2020 → H_Predicted_2019_2020.tif
2014–2015 → H_Predicted_2014_2015.tif
2024–2025 → H_Predicted_2024_2025.tif
1989–1990 → H_Predicted_1989_1990.tif
1994–1995 → H_Predicted_1994_1995

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# Paths
agb_mg_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_mg"

# Get all AGB mg files
agb_files = sorted(glob.glob(os.path.join(agb_mg_path, "AGB_Chave_mg_*.tif")))

# Prepare summary list
summary_list = []

for f in agb_files:
    year = os.path.basename(f).split("_")[-1].replace(".tif","")

    with rasterio.open(f) as src:
        data = src.read(1)
        data = data[data > 0]  # Mask out zeros/nodata

        # Convert mg/pixel → mg/ha
        data_mgha = data * 400

        summary_list.append({
            "Year": year,
            "Mean_mg_ha": np.mean(data_mgha),
            "Median_mg_ha": np.median(data_mgha),
            "Min_mg_ha": np.min(data_mgha),
            "Max_mg_ha": np.max(data_mgha),
            "Std_mg_ha": np.std(data_mgha)
        })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_list)
summary_df = summary_df.sort_values("Year").reset_index(drop=True)

# Display
print(summary_df)


   Year    Mean_mg_ha  Median_mg_ha     Min_mg_ha     Max_mg_ha     Std_mg_ha
0  1990  9.650202e+11  9.646329e+11  2.591864e+11  2.213421e+12  2.658168e+11
1  1995  9.627761e+11  9.572798e+11  2.524349e+11  1.989516e+12  2.490061e+11
2  2000  9.886273e+11  9.636651e+11  2.495807e+11  2.569950e+12  2.793863e+11
3  2005  5.925638e+11  2.536499e+11  1.375324e+11  1.942092e+12  3.955066e+11
4  2010  9.753562e+11  9.669132e+11  2.360929e+11  2.213947e+12  2.538073e+11
5  2015  7.907271e+11  6.199795e+11  3.811241e+11  2.985297e+12  2.590118e+11
6  2020  8.180441e+11  6.199795e+11  3.795671e+11  2.985297e+12  3.069270e+11
7  2024  8.078741e+11  6.199795e+11  3.799974e+11  2.985297e+12  2.887351e+11


Revise Chave Etal Equation

In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import glob
import os

# Paths
agb_kg_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg"
agb_utm_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_UTM"
os.makedirs(agb_utm_path, exist_ok=True)

# Get all AGB rasters
agb_files = sorted(glob.glob(os.path.join(agb_kg_path, "AGB_Chave_kg_*.tif")))

# Target CRS: UTM Arc 1960 / 35N
dst_crs = "EPSG:21095"

for f in agb_files:
    with rasterio.open(f) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        out_file = os.path.join(agb_utm_path, os.path.basename(f).replace(".tif", "_utm.tif"))

        with rasterio.open(out_file, "w", **kwargs) as dst:
            for i in range(1, src.count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src.transform,
                    src_crs=src.crs,
                    dst_transform=transform,
                    dst_crs=dst_crs,
                    resampling=Resampling.bilinear
                )

    print(f"✅ Reprojected {os.path.basename(f)} to UTM 21035")

print("🎉 All rasters reprojected successfully.")


✅ Reprojected AGB_Chave_kg_1990.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_1995.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2000.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2005.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2010.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2015.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2020.tif to UTM 21035
✅ Reprojected AGB_Chave_kg_2024.tif to UTM 21035
🎉 All rasters reprojected successfully.


In [ ]:
import numpy as np
import rasterio
import os
import glob

agb_utm_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_UTM"
agb_mgha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_Mg_ha_UTM"
os.makedirs(agb_mgha_path, exist_ok=True)

agb_files = sorted(glob.glob(os.path.join(agb_utm_path, "*_utm.tif")))

for f in agb_files:
    with rasterio.open(f) as src:
        data = src.read(1).astype("float32")
        transform = src.transform
        pixel_area = abs(transform.a * transform.e)  # m² per pixel
        data_mgha = data * (pixel_area / 10000) / 1000  # kg/pixel → Mg/ha
        meta = src.meta.copy()
        meta.update(dtype="float32", nodata=0)

        out_file = os.path.join(agb_mgha_path, os.path.basename(f).replace("_utm.tif", "_Mg_ha.tif"))

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(data_mgha, 1)

    print(f"✅ Converted {os.path.basename(f)} to Mg/ha using actual pixel area")


✅ Converted AGB_Chave_kg_1990_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_1995_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2000_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2005_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2010_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2015_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2020_utm.tif to Mg/ha using actual pixel area
✅ Converted AGB_Chave_kg_2024_utm.tif to Mg/ha using actual pixel area


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# =========================
# Paths
# =========================
agb_utm_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_UTM"
summary_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_summary_UTM"
os.makedirs(summary_path, exist_ok=True)

# Get all UTM AGB rasters
agb_files = sorted(glob.glob(os.path.join(agb_utm_path, "*_utm.tif")))

# =========================
# Prepare summary list
# =========================
summary_list = []

for f in agb_files:
    # Extract year correctly
    year = os.path.basename(f).split("_")[-2]  # AGB_Chave_kg_YYYY_utm.tif → [-2] is the year

    with rasterio.open(f) as src:
        data_kg = src.read(1).astype("float32")
        transform = src.transform

        # Pixel area in m²
        pixel_area = abs(transform.a * transform.e)

        # Convert kg/pixel → Mg/ha
        data_mgha = data_kg * (pixel_area / 10000) / 1000  # kg → Mg, pixel → ha

        # Mask invalid values
        data_mgha = data_mgha[data_mgha > 0]

        summary_list.append({
            "Year": year,
            "Mean_Mg_ha": np.mean(data_mgha),
            "Median_Mg_ha": np.median(data_mgha),
            "Min_Mg_ha": np.min(data_mgha),
            "Max_Mg_ha": np.max(data_mgha),
            "Std_Mg_ha": np.std(data_mgha)
        })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_list)
summary_df = summary_df.sort_values("Year").reset_index(drop=True)

# Save as CSV if needed
summary_df.to_csv(os.path.join(summary_path, "AGB_summary_Mg_ha_UTM.csv"), index=False)

# Display
print(summary_df)


   Year  Mean_Mg_ha  Median_Mg_ha  Min_Mg_ha  Max_Mg_ha  Std_Mg_ha
0  1990        0.01          0.01       0.00       0.01       0.00
1  1995        0.01          0.01       0.00       0.01       0.00
2  2000        0.01          0.01       0.00       0.02       0.00
3  2005        0.14          0.06       0.04       0.43       0.09
4  2010        0.01          0.01       0.00       0.01       0.00
5  2015        0.01          0.00       0.00       0.02       0.00
6  2020        0.01          0.00       0.00       0.02       0.00
7  2024        0.01          0.00       0.00       0.02       0.00


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# =========================
# Paths
# =========================
agb_utm_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_UTM"
agb_kg_ha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_ha_UTM"
os.makedirs(agb_kg_ha_path, exist_ok=True)

# Get all UTM AGB rasters
agb_files = sorted(glob.glob(os.path.join(agb_utm_path, "*_utm.tif")))

# =========================
# Convert kg/pixel → kg/ha
# =========================
for f in agb_files:
    with rasterio.open(f) as src:
        data_kg = src.read(1).astype("float32")
        transform = src.transform
        meta = src.meta.copy()

        # Pixel area in m²
        pixel_area = abs(transform.a * transform.e)

        # Convert kg/pixel → kg/ha
        data_kg_ha = data_kg * (10000 / pixel_area)

        # Update metadata
        meta.update(dtype="float32", nodata=0)

        # Save output
        year = os.path.basename(f).split("_")[-2]  # AGB_Chave_kg_YYYY_utm.tif → [-2] is year
        out_file = os.path.join(agb_kg_ha_path, f"AGB_Chave_kg_ha_{year}.tif")

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(data_kg_ha, 1)

    print(f" Converted {os.path.basename(f)} → kg/ha")

print(" All AGB rasters successfully converted to kg/ha (UTM projected).")


 Converted AGB_Chave_kg_1990_utm.tif → kg/ha
 Converted AGB_Chave_kg_1995_utm.tif → kg/ha
 Converted AGB_Chave_kg_2000_utm.tif → kg/ha
 Converted AGB_Chave_kg_2005_utm.tif → kg/ha
 Converted AGB_Chave_kg_2010_utm.tif → kg/ha
 Converted AGB_Chave_kg_2015_utm.tif → kg/ha
 Converted AGB_Chave_kg_2020_utm.tif → kg/ha
 Converted AGB_Chave_kg_2024_utm.tif → kg/ha
 All AGB rasters successfully converted to kg/ha (UTM projected).


In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# =========================
# Paths
# =========================
agb_kg_ha_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_Chave_kg_ha_UTM"
summary_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/AGB_summary_kg_ha_UTM"
os.makedirs(summary_path, exist_ok=True)

# Get all kg/ha rasters
agb_files = sorted(glob.glob(os.path.join(agb_kg_ha_path, "AGB_Chave_kg_ha_*.tif")))

# Prepare summary list
summary_list = []

for f in agb_files:
    # Extract year from filename
    year = os.path.basename(f).split("_")[-1].replace(".tif","")

    with rasterio.open(f) as src:
        data = src.read(1).astype("float32")
        data = data[data > 0]  # Mask out zeros/no-data

        summary_list.append({
            "Year": year,
            "Mean_kg_ha": np.mean(data),
            "Median_kg_ha": np.median(data),
            "Min_kg_ha": np.min(data),
            "Max_kg_ha": np.max(data),
            "Std_kg_ha": np.std(data)
        })

# Convert to DataFrame
summary_df = pd.DataFrame(summary_list)
summary_df = summary_df.sort_values("Year").reset_index(drop=True)

# Save as CSV if needed
summary_df.to_csv(os.path.join(summary_path, "AGB_summary_kg_ha_UTM.csv"), index=False)

# Display
print(summary_df)


   Year     Mean_kg_ha   Median_kg_ha      Min_kg_ha     Max_kg_ha  \
0  1990  948388.312500  949426.437500  265552.500000  2.174989e+06   
1  1995  946173.625000  941099.687500  258171.843750  1.949107e+06   
2  2000  971568.750000  947256.875000  262307.781250  2.517464e+06   
3  2005   16168.996094    6923.623535    4313.570801  5.126643e+04   
4  2010  958534.875000  950459.750000  237189.734375  2.175506e+06   
5  2015  776993.312500  609214.687500  380958.031250  2.736577e+06   
6  2020  803835.812500  609214.687500  378770.812500  2.725410e+06   
7  2024  793842.187500  609214.687500  380958.031250  2.739685e+06   

       Std_kg_ha  
0  257270.718750  
1  241206.843750  
2  271072.093750  
3   10533.281250  
4  245915.890625  
5  249808.015625  
6  297156.000000  
7  279170.031250  


In [ ]:
import os
import glob
import re

h_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted"

h_files = sorted(glob.glob(os.path.join(h_path, "H_Predicted_*.tif")))

for f in h_files:
    basename = os.path.basename(f)
    match = re.search(r"(\d{4})_(\d{4})", basename)
    if match:
        start, end = match.group(1), match.group(2)
        new_name = f"H_Predicted_{end}.tif"
        new_path = os.path.join(h_path, new_name)
        print(f"Renaming {basename} → {new_name}")
        os.rename(f, new_path)


In [ ]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import glob
import os

# Paths
h_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs"
utm_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs/UTM"
os.makedirs(utm_path, exist_ok=True)

# Get H files
h_files = sorted(glob.glob(os.path.join(h_path, "H_Predicted_*.tif")))
print("H files to reproject:", h_files)

# Reproject function
def reproject_to_utm(input_files, out_dir, suffix="_h_utm.tif", dst_crs="EPSG:21095"):
    utm_files = []
    for f in input_files:
        with rasterio.open(f) as src:
            transform, width, height = calculate_default_transform(
                src.crs, dst_crs, src.width, src.height, *src.bounds
            )
            kwargs = src.meta.copy()
            kwargs.update({
                "crs": dst_crs,
                "transform": transform,
                "width": width,
                "height": height,
                "dtype": "float32"
            })

            out_file = os.path.join(out_dir, os.path.basename(f).replace(".tif", suffix))
            with rasterio.open(out_file, "w", **kwargs) as dst:
                for i in range(1, src.count + 1):
                    reproject(
                        source=rasterio.band(src, i),
                        destination=rasterio.band(dst, i),
                        src_transform=src.transform,
                        src_crs=src.crs,
                        dst_transform=transform,
                        dst_crs=dst_crs,
                        resampling=Resampling.bilinear
                    )
            utm_files.append(out_file)
            print(f"✅ Reprojected {os.path.basename(f)} → {os.path.basename(out_file)}")
    return utm_files

# Reproject H files
h_utm_files = reproject_to_utm(h_files, utm_path)

# Check
print("\nAll reprojected H files saved in UTM folder:")
for f in h_utm_files:
    print(f)


H files to reproject: ['/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_1989_1990.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_1990_Masked.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_1994_1995.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_1995_Masked.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_1999_2000.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2000_Masked.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2004_2005.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2005_Masked.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2009_2010.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2010_Masked.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2014_2015.tif', '/content/drive/MyDrive/Kamatira Model Data/Outputs/H_Predicted_2015_Masked.tif', '/con

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import os
import glob
import re

# =========================
# Paths
# =========================
base_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs"
utm_path = os.path.join(base_path, "UTM")
agb_kg_ha_path = os.path.join(base_path, "AGB_Chave_kg_ha_UTM")
os.makedirs(agb_kg_ha_path, exist_ok=True)

# =========================
# Get UTM-reprojected DBH and H files
# =========================
dbh_files = sorted(glob.glob(os.path.join(utm_path, "*_dbh_utm.tif")))
h_files = sorted(glob.glob(os.path.join(utm_path, "*_h_utm.tif")))

# =========================
# Parse H files into year ranges
# =========================
h_ranges = []
for f in h_files:
    match = re.search(r"(\d{4})_(\d{4})", os.path.basename(f))
    if match:
        start, end = int(match.group(1)), int(match.group(2))
        h_ranges.append((start, end, f))

# =========================
# AGB calculation loop
# =========================
summary_list = []

for dbh_file in dbh_files:
    # Extract DBH year
    match = re.search(r"DBH_Predicted_(\d{4})", os.path.basename(dbh_file))
    if not match:
        print(f"⚠️ Skipped DBH file (no year): {os.path.basename(dbh_file)}")
        continue
    dbh_year = int(match.group(1))

    # Find matching H file
    h_file = None
    for start, end, hf in h_ranges:
        if start <= dbh_year <= end:
            h_file = hf
            break
    if h_file is None:
        print(f"⚠️ No matching H file for DBH year {dbh_year}")
        continue

    # =========================
    # Read rasters
    # =========================
    with rasterio.open(dbh_file) as dbh_src, rasterio.open(h_file) as h_src:
        D = dbh_src.read(1).astype("float32")  # DBH in cm
        H = h_src.read(1).astype("float32")   # Height in m
        meta = dbh_src.meta.copy()

        # Valid mask
        mask = (D > 0) & (H > 0)

        # =========================
        # Chave et al. 2009, 2014 AGB equation (kg/pixel)
        # =========================
        AGB_kg = np.zeros_like(D, dtype="float32")
        AGB_kg[mask] = 0.0673 * (0.598 * (D[mask]**2) * H[mask])**0.976

        # =========================
        # Convert kg/pixel → kg/ha using pixel area
        # =========================
        pixel_area = abs(dbh_src.transform.a * dbh_src.transform.e)  # m² per pixel
        AGB_kg_ha = AGB_kg * (10000 / pixel_area)

        # Save kg/ha raster
        out_file = os.path.join(agb_kg_ha_path, f"AGB_Chave_kg_ha_{dbh_year}.tif")
        meta.update(dtype="float32", nodata=0)
        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(AGB_kg_ha, 1)

    # =========================
    # Summary statistics
    # =========================
    data = AGB_kg_ha[mask]
    summary_list.append({
        "Year": dbh_year,
        "Mean_kg_ha": np.mean(data),
        "Median_kg_ha": np.median(data),
        "Min_kg_ha": np.min(data),
        "Max_kg_ha": np.max(data),
        "Std_kg_ha": np.std(data)
    })

    print(f"✅ Processed AGB for DBH year {dbh_year}")

# =========================
# Save summary table
# =========================
summary_df = pd.DataFrame(summary_list).sort_values("Year").reset_index(drop=True)
summary_csv = os.path.join(base_path, "AGB_summary_kg_ha_UTM.csv")
summary_df.to_csv(summary_csv, index=False)

print("🎉 AGB calculation and summary statistics completed successfully!")
print(summary_df)


✅ Processed AGB for DBH year 1990
✅ Processed AGB for DBH year 1995
✅ Processed AGB for DBH year 2000
✅ Processed AGB for DBH year 2005
✅ Processed AGB for DBH year 2010
✅ Processed AGB for DBH year 2015
✅ Processed AGB for DBH year 2020
✅ Processed AGB for DBH year 2024
🎉 AGB calculation and summary statistics completed successfully!
   Year     Mean_kg_ha   Median_kg_ha      Min_kg_ha     Max_kg_ha  \
0  1990  948027.562500  948948.562500  265482.937500  2.174989e+06   
1  1995  945850.187500  940688.250000  258298.000000  1.949081e+06   
2  2000  971239.312500  946851.062500  262223.468750  2.517441e+06   
3  2005   16119.604492    6923.623535    4326.985840  5.124471e+04   
4  2010  958207.750000  950014.687500  237069.578125  2.175506e+06   
5  2015  776606.437500  609214.687500  380958.031250  2.724942e+06   
6  2020  803407.000000  609214.687500  378732.812500  2.703761e+06   
7  2024  793420.687500  609214.687500  380958.031250  2.723570e+06   

       Std_kg_ha  
0  257243.468

In [ ]:
import rasterio
import numpy as np
import pandas as pd
import glob
import os

# =========================
# Paths
# =========================
base_path = "/content/drive/MyDrive/Kamatira Model Data/Outputs"
utm_path = os.path.join(base_path, "UTM")  # Reprojected rasters
agb_kg_ha_path = os.path.join(base_path, "AGB_Chave_kg_ha_UTM")
os.makedirs(agb_kg_ha_path, exist_ok=True)

# =========================
# Get reprojected DBH and H files
# =========================
dbh_files = sorted(glob.glob(os.path.join(utm_path, "DBH_Predicted_*_dbh_utm.tif")))
h_files = sorted(glob.glob(os.path.join(utm_path, "H_Predicted_*_h_utm.tif")))

# =========================
# Parse H files into year ranges
# =========================
h_ranges = []
for f in h_files:
    match = re.search(r"(\d{4})_(\d{4})", os.path.basename(f))
    if match:
        start, end = int(match.group(1)), int(match.group(2))
        h_ranges.append((start, end, f))

# =========================
# AGB calculation loop
# =========================
summary_list = []

for dbh_file in sorted(dbh_files):
    match = re.search(r"DBH_Predicted_(\d{4})_cm", os.path.basename(dbh_file))
    if not match:
        print(f"⚠️ Skipped DBH file (no year): {os.path.basename(dbh_file)}")
        continue
    dbh_year = int(match.group(1))

    # Find matching H file
    h_file = None
    for start, end, hf in h_ranges:
        if start <= dbh_year <= end:
            h_file = hf
            break
    if h_file is None:
        print(f"⚠️ No matching H file for DBH year {dbh_year}")
        continue

    with rasterio.open(dbh_file) as dbh_src, rasterio.open(h_file) as h_src:
        D = dbh_src.read(1).astype("float32")  # DBH in cm
        H = h_src.read(1).astype("float32")    # Height in m
        meta = dbh_src.meta.copy()

        mask = (D > 0) & (H > 0)

        # Chave et al. 2009 AGB equation (kg/pixel)
        AGB_kg = np.zeros_like(D, dtype="float32")
        AGB_kg[mask] = 0.0673 * (0.598 * (D[mask]**2) * H[mask])**0.976

        # Pixel area in m²
        pixel_area = abs(dbh_src.transform.a * dbh_src.transform.e)

        # Convert kg/pixel → kg/ha
        AGB_kg_ha = AGB_kg * (10000 / pixel_area)  # 10,000 m²/ha divided by pixel area

        # Save kg/ha raster
        out_file = os.path.join(agb_kg_ha_path, f"AGB_Chave_kg_ha_{dbh_year}.tif")
        meta.update(dtype="float32", nodata=0)
        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(AGB_kg_ha, 1)

        # Summary statistics
        data = AGB_kg_ha[mask]
        summary_list.append({
            "Year": dbh_year,
            "Mean_kg_ha": np.mean(data),
            "Median_kg_ha": np.median(data),
            "Min_kg_ha": np.min(data),
            "Max_kg_ha": np.max(data),
            "Std_kg_ha": np.std(data)
        })

    print(f"✅ Processed AGB for DBH year {dbh_year}")

# Convert summary to DataFrame
summary_df1 = pd.DataFrame(summary_list).sort_values("Year").reset_index(drop=True)

# Save CSV
summary_csv = os.path.join(base_path, "AGB_summary_kg_ha_UTM.csv")
summary_df1.to_csv(summary_csv, index=False)

print("🎉 AGB calculation and summary statistics completed successfully!")
print(summary_df1)


✅ Processed AGB for DBH year 1990
✅ Processed AGB for DBH year 1995
✅ Processed AGB for DBH year 2000
✅ Processed AGB for DBH year 2005
✅ Processed AGB for DBH year 2010
✅ Processed AGB for DBH year 2015
✅ Processed AGB for DBH year 2020
✅ Processed AGB for DBH year 2024
🎉 AGB calculation and summary statistics completed successfully!
   Year     Mean_kg_ha   Median_kg_ha      Min_kg_ha     Max_kg_ha  \
0  1990  948027.562500  948948.562500  265482.937500  2.174989e+06   
1  1995  945850.187500  940688.250000  258298.000000  1.949081e+06   
2  2000  971239.312500  946851.062500  262223.468750  2.517441e+06   
3  2005   16119.604492    6923.623535    4326.985840  5.124471e+04   
4  2010  958207.750000  950014.687500  237069.578125  2.175506e+06   
5  2015  776606.437500  609214.687500  380958.031250  2.724942e+06   
6  2020  803407.000000  609214.687500  378732.812500  2.703761e+06   
7  2024  793420.687500  609214.687500  380958.031250  2.723570e+06   

       Std_kg_ha  
0  257243.468

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:,.2f}'.format)  # Show floats with 2 decimals, no scientific notation

# Now display your summary
print(summary_df)


   Year  Mean_kg_ha  Median_kg_ha  Min_kg_ha    Max_kg_ha  Std_kg_ha
0  1990  948,027.56    948,948.56 265,482.94 2,174,988.75 257,243.47
1  1995  945,850.19    940,688.25 258,298.00 1,949,080.75 241,209.42
2  2000  971,239.31    946,851.06 262,223.47 2,517,440.75 271,037.72
3  2005   16,119.60      6,923.62   4,326.99    51,244.71  10,503.82
4  2010  958,207.75    950,014.69 237,069.58 2,175,506.00 245,923.05
5  2015  776,606.44    609,214.69 380,958.03 2,724,941.50 248,940.52
6  2020  803,407.00    609,214.69 378,732.81 2,703,760.75 296,431.38
7  2024  793,420.69    609,214.69 380,958.03 2,723,570.50 278,379.31


In [ ]:
import rasterio

# Example: check first DBH UTM raster
dbh_utm_file = "/content/drive/MyDrive/Kamatira Model Data/Outputs/UTM/DBH_Predicted_1990_cm_dbh_utm.tif"

with rasterio.open(dbh_utm_file) as src:
    transform = src.transform
    width = src.width
    height = src.height
    crs = src.crs

    # Pixel size in meters
    pixel_width = transform.a
    pixel_height = -transform.e  # usually negative in north-up images
    pixel_area = pixel_width * pixel_height

    print(f"Raster CRS: {crs}")
    print(f"Raster size: {width} x {height} pixels")
    print(f"Pixel size: {pixel_width:.2f} m x {pixel_height:.2f} m")
    print(f"Pixel area: {pixel_area:.2f} m²")
    print(f"Scaling factor to convert kg/pixel → kg/ha: {10000 / pixel_area:.2f}")


Raster CRS: EPSG:21095
Raster size: 2094 x 853 pixels
Pixel size: 5.04 m x 5.04 m
Pixel area: 25.44 m²
Scaling factor to convert kg/pixel → kg/ha: 393.05


In [ ]:
import geopandas as gpd
from rasterio.mask import mask

# 🔹 Paths
predicted_path = "/content/drive/MyDrive/Kamatira Model Data/Forest/Clipped/simulated_2025_RF.tif"
clip_output_path = "/content/drive/MyDrive/Kamatira Model Data/Forest/Predictions/Clipped_simulated_2025_RF.tif"
vector_path = "/content/drive/MyDrive/Kamatira Model Data/GeoFiles/Kamatira.geojson"

# 🔹 Read vector
gdf = gpd.read_file(vector_path)
gdf = gdf.to_crs("EPSG:4326")  # Optional: force CRS match if needed

# 🔹 Read and clip raster
with rasterio.open(predicted_path) as src:
    out_image, out_transform = mask(src, gdf.geometry, crop=True)
    out_meta = src.meta.copy()

out_meta.update({
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

# 🔹 Save clipped raster
with rasterio.open(clip_output_path, "w", **out_meta) as dest:
    dest.write(out_image)

print(f" Clipped prediction saved to: {clip_output_path}")
